# Image→Image Evaluation — Notebook

Two end-to-end scenarios — all figures are interactive (Plotly):

- **Part 1 — Basic** (`emb_image2image.json`) — class labels derived from the path hierarchy.
- **Part 2 — With Metadata** (`emb_image2image_meta.json`) — explicit `MetadataGroup` clusters enabling graded nDCG and per-attribute KPIs.

Run all cells top-to-bottom.

In [1]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

from precisionai.agrieval.emb.schemas.evaluate import MetadataGroup
from precisionai.agrieval.emb.services.evaluate import run_image2image_eval
from precisionai.agrieval.emb.services.reporting import (
    plot_cosine_similarity,
    plot_knn_confusion,
    plot_lle,
    plot_tsne,
    print_result,
)

---
## Part 1 — Basic

`emb_image2image.json` contains 20 embeddings across two classes (A and D), with class labels derived from the image path hierarchy.

### Load Data

In [2]:
payload_path = Path("emb_image2image.json")
if not payload_path.exists():
    raise FileNotFoundError("emb_image2image.json not found — start Jupyter from the examples/ directory")

with open(payload_path) as f:
    payload = json.load(f)

embeddings: dict[str, list[float]] = payload["embeddings"]
print(f"Loaded {len(embeddings)} embeddings  (dim={len(next(iter(embeddings.values())))})")

Loaded 20 embeddings  (dim=32)


### Run Evaluation

In [3]:
result = run_image2image_eval(
    image_embeddings=embeddings,
    k_values=[5, 10],
    dataset_root="images",
    sample_pairs=None,
)

print_result(result)

n_items      : 20
embedding_dim: 32
classes      : ['A', 'D']
k_values     : [5, 10]

── global_metrics ──────────────────────────────────────────────────────
  pairwise cosine    : mean=0.1968  std=0.4371  (p05=-0.4400  p50=0.1563  p95=0.9423)
  centroid cosine    : mean=0.4868  std=0.2018  norm=0.4868
  intra/inter gap    : 0.5274  (intra=0.4744  inter=-0.0530)
  effective_rank     : 3.02  (ratio=0.0943  dim=32)
  uniformity         : -1.6982

  hubness@5         : mean=5.0000  std=2.0000  p95=10.0500
  hubness@10        : mean=10.0000  std=4.9800  p95=18.0500
  knn_radius@5         : mean=0.2105  std=0.1204  p05=-0.0166  p95=0.3442
  knn_radius@10        : mean=0.1266  std=0.1223  p05=-0.1054  p95=0.2498
  mean_top_k_sim@5         : mean=0.7896  std=0.0277  p05=0.7380  p95=0.8137
  mean_top_k_sim@10        : mean=0.4782  std=0.0745  p05=0.3320  p95=0.5384
  outlier_score@5         : mean=0.2104  std=0.0277  p95=0.2620
  outlier_score@10        : mean=0.5218  std=0.0745  p95=0.6680



### Visualizations

All plots are interactive — hover for details, click legend entries to toggle classes, and drag to rotate 3D views.

#### KNN Confusion Matrix

Rows = true class, columns = neighbor class, values = fraction of k-NN neighbors belonging to each class. The diagonal equals mean KNN purity — higher is better.

In [4]:
plot_knn_confusion(result, output_path=None)

#### Pairwise Cosine Similarity

Full N×N cosine similarity matrix sorted by class. Within-class blocks sit on the diagonal — tighter, brighter blocks indicate a more discriminative embedding space.

In [5]:
plot_cosine_similarity(embeddings, result, output_path=None)

#### t-SNE — 2D

t-SNE preserves local neighborhood structure. Well-separated clusters indicate the model has learned class-discriminative features.

In [6]:
plot_tsne(embeddings, result, output_path=None, dimensions=2)

  t-SNE 2D — fitting 20 samples (perplexity=3, iter=1000)...


#### t-SNE — 3D

3D variant — drag to rotate, scroll to zoom.

In [7]:
plot_tsne(embeddings, result, output_path=None, dimensions=3)

  t-SNE 3D — fitting 20 samples (perplexity=3, iter=2000)...


#### LLE — 3D

Locally Linear Embedding preserves local geometry rather than global distances, complementing the t-SNE view. Drag to rotate.

In [8]:
plot_lle(embeddings, result, output_path=None)

  LLE 3D — fitting 20 samples (n_neighbors=3)...


---
## Part 2 — With Metadata

`emb_image2image_meta.json` supplies a `metadata` field with explicit `L2` clusters (A1, A2, D1, D2), enabling graded nDCG and per-attribute KPIs.

### Load Data

In [9]:
payload_path = Path("emb_image2image_meta.json")
if not payload_path.exists():
    raise FileNotFoundError("emb_image2image_meta.json not found — start Jupyter from the examples/ directory")

with open(payload_path) as f:
    payload = json.load(f)

embeddings: dict[str, list[float]] = payload["embeddings"]
metadata: dict[str, MetadataGroup] = {key: MetadataGroup(**group) for key, group in payload["metadata"].items()}

print(f"Loaded {len(embeddings)} embeddings  (dim={len(next(iter(embeddings.values())))})")
print(f"Metadata groups: {list(metadata.keys())}")

Loaded 20 embeddings  (dim=32)
Metadata groups: ['A1', 'A2', 'D1', 'D2']


### Run Evaluation

In [10]:
result = run_image2image_eval(
    image_embeddings=embeddings,
    k_values=[5, 10],
    dataset_root="images",
    sample_pairs=None,
    metadata=metadata,
)

print_result(result)

n_items      : 20
embedding_dim: 32
classes      : ['A', 'D']
k_values     : [5, 10]

── global_metrics ──────────────────────────────────────────────────────
  pairwise cosine    : mean=0.1968  std=0.4371  (p05=-0.4400  p50=0.1563  p95=0.9423)
  centroid cosine    : mean=0.4868  std=0.2018  norm=0.4868
  intra/inter gap    : 0.5274  (intra=0.4744  inter=-0.0530)
  effective_rank     : 3.02  (ratio=0.0943  dim=32)
  uniformity         : -1.6982
  alignment          : 0.1313

  hubness@5         : mean=5.0000  std=2.0000  p95=10.0500
  hubness@10        : mean=10.0000  std=4.9800  p95=18.0500
  knn_radius@5         : mean=0.2105  std=0.1204  p05=-0.0166  p95=0.3442
  knn_radius@10        : mean=0.1266  std=0.1223  p05=-0.1054  p95=0.2498
  mean_top_k_sim@5         : mean=0.7896  std=0.0277  p05=0.7380  p95=0.8137
  mean_top_k_sim@10        : mean=0.4782  std=0.0745  p05=0.3320  p95=0.5384
  outlier_score@5         : mean=0.2104  std=0.0277  p95=0.2620
  outlier_score@10        : mean=0.

### Visualizations

All plots are interactive — hover for details, click legend entries to toggle classes, and drag to rotate 3D views.

#### KNN Confusion Matrix

Rows = true class, columns = neighbor class, values = fraction of k-NN neighbors belonging to each class. The diagonal equals mean KNN purity — higher is better.

In [11]:
plot_knn_confusion(result, output_path=None)

#### Pairwise Cosine Similarity

Full N×N cosine similarity matrix sorted by class. Within-class blocks sit on the diagonal — tighter, brighter blocks indicate a more discriminative embedding space.

In [12]:
plot_cosine_similarity(embeddings, result, output_path=None)

#### t-SNE — 2D

t-SNE preserves local neighborhood structure. Well-separated clusters indicate the model has learned class-discriminative features.

In [13]:
plot_tsne(embeddings, result, output_path=None, dimensions=2)

  t-SNE 2D — fitting 20 samples (perplexity=3, iter=1000)...


#### t-SNE — 3D

3D variant — drag to rotate, scroll to zoom.

In [14]:
plot_tsne(embeddings, result, output_path=None, dimensions=3)

  t-SNE 3D — fitting 20 samples (perplexity=3, iter=2000)...


#### LLE — 3D

Locally Linear Embedding preserves local geometry rather than global distances, complementing the t-SNE view. Drag to rotate.

In [15]:
plot_lle(embeddings, result, output_path=None)

  LLE 3D — fitting 20 samples (n_neighbors=3)...
